In [1]:
import torch
import gc
import time

# Unload models and clean up gpu memory cache
def free_gpu(model):
  if model:
    # Removes the reference to the model's memory,
    # making it eligible for garbage collection.
    del model

  # Release any cached GPU memory that's no longer needed.
  if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

  # Trigger garbage collection to ensure memory is fully released.
  gc.collect()

free_gpu(None)

In [2]:
import os
import sys
from pathlib import Path

os.environ["PYTHONNOUSERSITE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

sys.path.insert(0, str(Path.cwd().parent if (Path.cwd() / "llm").exists() else Path.cwd()))
from local_paths import model_path

MODEL_PATH = str(model_path("QWEN_MODEL"))

import torch
import vllm

print("Torch:", torch.__version__)
print("vLLM:", vllm.__version__)
print("MODEL exists:", Path(MODEL_PATH).exists())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

<USER_SITE>/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-09-13 11:20:18,829	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Torch: 2.4.0+cu118
vLLM: 0.6.1.post1
GPU: Tesla V100-SXM2-32GB


In [3]:
from typing import List

from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel

from vllm import LLM, SamplingParams

In [4]:
llm = LLM(
    model=MODEL_PATH,
    dtype="float16",
    gpu_memory_utilization=0.15,
    max_model_len=1024,
    max_num_seqs=4,
    enforce_eager=True,
)

WARNING 09-13 11:20:20 config.py:1657] Casting torch.bfloat16 to torch.float16.
WARNING 09-13 11:20:20 config.py:383] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 09-13 11:20:20 llm_engine.py:223] Initializing an LLM engine (v0.6.1.post1) with config: model='<MODELS_DIR>/Qwen2.5-0.5B-Instruct', speculative_config=None, tokenizer='<MODELS_DIR>/Qwen2.5-0.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observabili

<CONDA_ENV>/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
<CONDA_ENV>/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")


INFO 09-13 11:20:22 model_runner.py:997] Starting to load model <MODELS_DIR>/Qwen2.5-0.5B-Instruct...
INFO 09-13 11:20:22 selector.py:217] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
INFO 09-13 11:20:22 selector.py:116] Using XFormers backend.


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  3.92it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  3.91it/s]



INFO 09-13 11:20:23 model_runner.py:1008] Loading model weights took 0.9276 GB
INFO 09-13 11:20:24 gpu_executor.py:122] # GPU blocks: 16699, # CPU blocks: 21845


In [5]:
from fastapi import FastAPI, Depends
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from llm import LLMEngine
from llm.flow import flow, flow_banner, flow_done, flow_skip
from typing import List
import asyncio
import multiprocessing
import atexit

# Architecture path (what FLOW prints trace):
# Client → FastAPI → LLMEngine → WorkloadManager → ModelExecutor → ModelWorker → ModelManager/model → GPU
#
# Reuses the earlier `llm = LLM(MODEL_PATH, ...)` for /generate_vllm.
# ModelWorker loads the same MODEL_PATH via transformers for the custom stack.

app = FastAPI(title="Single Model LLM Serving (architecture demo)")

_llm = None
_llm_lock = multiprocessing.Lock()

def cleanup():
    global _llm
    if _llm is not None:
        try:
            _llm._cleanup()
        except Exception:
            pass
        _llm = None

def get_llm():
    global _llm
    with _llm_lock:
        if _llm is None:
            flow("FastAPI", f"get_llm() — creating LLMEngine(model_path={MODEL_PATH!r})")
            _llm = LLMEngine(
                model_path=MODEL_PATH,
                vllm_model=llm,  # reuse LLM(MODEL_PATH) from previous cell
            )
            atexit.register(cleanup)
        return _llm

class GenerateRequest(BaseModel):
    prompt: str

class GenerateResponse(BaseModel):
    generated_text: str

class BatchGenerateRequest(BaseModel):
    prompts: List[str]

class BatchGenerateResponse(BaseModel):
    generated_texts: List[str]

# Eager-init so ModelWorker / ModelManager startup prints appear before first request
engine = get_llm()
print("LLMEngine ready — using Qwen at", MODEL_PATH)

>>> [FastAPI] get_llm() — creating LLMEngine(model_path='<MODELS_DIR>/Qwen2.5-0.5B-Instruct')
>>> [LLMEngine] __init__ — model_path='<MODELS_DIR>/Qwen2.5-0.5B-Instruct'
>>> [ModelExecutor] __init__ — spawn queues created (safe with CUDA/vLLM in parent)
>>> [WorkloadManager] __init__ — queues ready (batch_size=4)
>>> [LLMEngine] setup_worker('<MODELS_DIR>/Qwen2.5-0.5B-Instruct') — starts ModelWorker process
>>> [ModelExecutor] setup_worker('<MODELS_DIR>/Qwen2.5-0.5B-Instruct') — spawn ModelWorker process
>>> [ModelExecutor] worker process started pid=44888 — waiting for ready…
>>> [ModelWorker] run() — spawn worker started, loading model…
>>> [ModelWorker] __init__ — device=cuda, loading model via ModelManager
>>> [ModelManager] __init__ — model_dir=model_cache
>>> [ModelManager] load_model('<MODELS_DIR>/Qwen2.5-0.5B-Instruct') — called at worker startup (not per request)
>>> [ModelManager] from_pretrained(model) local_files_only=True dtype=torch.float16
>>> [ModelManager] from_pretrain

In [6]:
@app.post("/basic_generate", response_model=GenerateResponse)
async def basic_generate(request: GenerateRequest, llm: LLMEngine = Depends(get_llm)):
    flow_banner("POST /basic_generate")
    flow("FastAPI", f"/basic_generate — prompt={request.prompt!r}")
    flow("FastAPI", "→ LLMEngine.basic_generate()")
    flow_skip("WorkloadManager", "basic_generate bypasses queue/batching; goes straight to ModelExecutor")

    generated_text = llm.basic_generate(request.prompt)

    flow("FastAPI", "← returning GenerateResponse")
    flow_done("POST /basic_generate")
    return GenerateResponse(generated_text=generated_text)


@app.post("/generate_stream")
async def generate_stream(request: GenerateRequest, llm: LLMEngine = Depends(get_llm)):
    flow_banner("POST /generate_stream")
    flow("FastAPI", f"/generate_stream — prompt={request.prompt!r}")
    flow("FastAPI", "→ LLMEngine.event_generator()  [uses WorkloadManager + streaming loop]")

    async def event_generator():
        loop = asyncio.get_event_loop()
        async for token in llm.event_generator(loop, request.prompt):
            yield token
        flow_done("POST /generate_stream")

    return StreamingResponse(event_generator(), media_type="text/event-stream")

In [7]:
@app.post("/generate", response_model=BatchGenerateResponse)
async def generate(request: BatchGenerateRequest, llm: LLMEngine = Depends(get_llm)):
    flow_banner("POST /generate")
    flow("FastAPI", f"/generate — {len(request.prompts)} prompts")
    flow("FastAPI", "→ LLMEngine.generate()  [uses WorkloadManager batching]")

    generated_texts = llm.generate(request.prompts)

    flow("FastAPI", "← returning BatchGenerateResponse")
    flow_done("POST /generate")
    return BatchGenerateResponse(generated_texts=generated_texts)


@app.post("/generate_vllm", response_model=BatchGenerateResponse)
async def generate_vllm(request: BatchGenerateRequest, llm: LLMEngine = Depends(get_llm)):
    flow_banner("POST /generate_vllm")
    flow("FastAPI", f"/generate_vllm — {len(request.prompts)} prompts")
    flow("FastAPI", "→ LLMEngine.generate_vllm()")
    flow_skip("WorkloadManager", "vLLM path does not use WorkloadManager")
    flow_skip("ModelExecutor", "vLLM path does not use ModelExecutor")
    flow_skip("ModelWorker", "vLLM path does not use ModelWorker")
    flow_skip("ModelManager", "vLLM loads/runs its own engine")

    generated_texts = llm.generate_vllm(request.prompts)

    flow("FastAPI", "← returning BatchGenerateResponse")
    flow_done("POST /generate_vllm")
    return BatchGenerateResponse(generated_texts=generated_texts)

In [ ]:
import threading
import uvicorn

def run_server():
    uvicorn.run(
        app,
        host="127.0.0.1",
        port=8000,
        log_level="info"
    )

server_thread = threading.Thread(
    target=run_server,
    daemon=True
)

server_thread.start()

INFO:     Started server process [44527]


INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)



 REQUEST  POST /basic_generate
>>> [FastAPI] /basic_generate — prompt='What is the capital of the United States?'
>>> [FastAPI] → LLMEngine.basic_generate()
--- SKIP [WorkloadManager] basic_generate bypasses queue/batching; goes straight to ModelExecutor
>>> [LLMEngine] basic_generate(prompt='What is the capital of the United States?')
--- SKIP [WorkloadManager] basic_generate builds Sequence locally; no add_request / get_next_batch
>>> [LLMEngine] → ModelExecutor.execute_batch([sequence])
>>> [ModelExecutor] execute_batch — sending 1 item(s) to worker (is_streaming=False)
>>> [ModelExecutor] waiting on result_queue…
>>> [ModelWorker] task received — is_streaming=False, batch_size=1
>>> [ModelWorker] generate() — batch size=1
--- SKIP [ModelManager] model already loaded at worker startup; not reloaded per request
>>> [GPU] model.generate() on cuda  input_shape=(1, 9)


Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


>>> [GPU] ← model.generate() finished
>>> [ModelWorker] generate() — returning 1 result(s)
>>> [ModelWorker] result put on result_queue
>>> [ModelExecutor] ← result received from ModelWorker
>>> [LLMEngine] ← result received from ModelExecutor
>>> [FastAPI] ← returning GenerateResponse
 DONE     POST /basic_generate

INFO:     127.0.0.1:41812 - "POST /basic_generate HTTP/1.1" 200 OK

 REQUEST  POST /generate
>>> [FastAPI] /generate — 4 prompts
>>> [FastAPI] → LLMEngine.generate()  [uses WorkloadManager batching]
>>> [LLMEngine] generate(4 prompts) — uses WorkloadManager
>>> [WorkloadManager] add_request → incoming_queue  id=7a0a61c3… prompt='The capital of France is'
>>> [WorkloadManager] add_request → incoming_queue  id=006cff72… prompt='The capital of India is'
>>> [WorkloadManager] add_request → incoming_queue  id=b0c77899… prompt='The capital of Japan is'
>>> [WorkloadManager] add_request → incoming_queue  id=d46619e1… prompt='The largest planet is'
>>> [WorkloadManager] get_next_b